# AutoGluon 二级资本债历史摘录
只保留二级资本债单元及共享合并函数。各段标签不同，不可依次执行作为统一实验。


原始 basis_ml.ipynb 单元索引 6（从 0 开始）

In [4]:
import os
import re
import pandas as pd

def merge_excels_by_first_col(folder_path, output_file="merged.xlsx"):
    # 排除：以8位日期开头、紧跟“-因子清单”的文件（不含扩展名匹配）
    pattern = re.compile(r"^\d{8}-因子清单")

    files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".xlsx")
        and not f.startswith("~")
        and not pattern.match(os.path.splitext(f)[0])
    ]
    if not files:
        print("未找到任何可用的 .xlsx 文件")
        return

    dfs = []
    for f in sorted(files):
        path = os.path.join(folder_path, f)
        try:
            # 第一行是列名，第一列作为索引（合并键）
            df = pd.read_excel(path, header=0, index_col=0)
        except Exception as e:
            print(f"读取失败：{path}，原因：{e}")
            continue

        # 规范化索引（第一列），避免空白/类型不一致
        df.index = df.index.astype(str).str.strip()

        # 列名加前缀（保留原始列名，避免冲突）
        stem = os.path.splitext(f)[0]
        df.columns = [f"{stem}__{col}" for col in df.columns]

        dfs.append(df)

    if not dfs:
        print("没有成功读取的表格")
        return

    # 以索引（第一列）按“交集”对齐拼接
    merged = pd.concat(dfs, axis=1, join="inner")

    # 索引转为普通列，并命名为 "date"
    merged = merged.reset_index().rename(columns={"index": "date"})

    # 导出
    merged.to_excel(output_file, index=False, header=True)
    print(f"合并完成，结果已保存到 {output_file}")
    print(f"合并后形状：{merged.shape}（行=共有键数量，列=1+各文件数据列总数）")

# 用法
merge_excels_by_first_col(r"factor", output_file="merged.xlsx")


合并完成，结果已保存到 merged.xlsx
合并后形状：(2122, 771)（行=共有键数量，列=1+各文件数据列总数）


原始 basis_ml.ipynb 单元索引 29（从 0 开始）

In [1]:
import data
import new_factor as nf
import numpy as np
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
import factor
import re
import os, sys
from datetime import datetime

d:\conda\envs\pavane\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


原始 basis_ml.ipynb 单元索引 30（从 0 开始）

In [10]:
myfactor.get_data('tier2').to_excel('factor/tier2.xlsx')
df_raw = merge_excels_by_first_col('factor', 'merged.xlsx')

原始 basis_ml.ipynb 单元索引 31（从 0 开始）

In [3]:
df_raw = pd.read_excel('merged_tier2.xlsx')
df_raw.set_index('date', inplace=True)

# 涨跌标签：下一期比当前期涨为 1，跌为 -1，不变为 0
df_raw['label'] = (df_raw['tier2__二级资本债_5Y'].shift(-1) - df_raw['tier2__二级资本债_5Y']).apply(
    lambda x: 1 if x > 0 else -1
)
# 删除最后一行（因为没有未来数据做标签）
df_clean = df_raw.iloc[:-1]
# 假设 df_clean 已经准备好，索引为时间或顺序

label_col = "label"
split_point = int(len(df_clean) * 0.8)

# 按顺序切分
df_train = df_clean.iloc[:split_point]
df_test  = df_clean.iloc[split_point:]

# 拆分特征和标签
X_train, y_train = df_train.drop(columns=[label_col]), df_train[label_col]
X_test,  y_test  = df_test.drop(columns=[label_col]),  df_test[label_col]

train_data = pd.concat([X_train,y_train],axis=1)
test_data = pd.concat([X_test, y_test],axis = 1)

原始 basis_ml.ipynb 单元索引 32（从 0 开始）

## 训练

原始 basis_ml.ipynb 单元索引 33（从 0 开始）

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor
predictor = TabularPredictor(label=label_col).fit(train_data)

No path specified. Models will be saved in: "AutogluonModels\ag-20251015_014109"
Preset alias specified: 'best' maps to 'best_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          16
Memory Avail:       3.33 GB / 15.73 GB (21.1%)
Disk Space Avail:   69.09 GB / 274.71 GB (25.2%)
Presets specified: ['best']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify t

原始 basis_ml.ipynb 单元索引 34（从 0 开始）

In [18]:
predictor.evaluate(test_data,auxiliary_metrics=True)

{'accuracy': 0.5023474178403756,
 'balanced_accuracy': np.float64(0.40138952511833875),
 'mcc': 0.16404065590559166}

原始 basis_ml.ipynb 单元索引 35（从 0 开始）

In [20]:
from sklearn.metrics import classification_report, roc_auc_score

y_true = test_data["label"]
y_pred = predictor.predict(test_data)
y_prob = predictor.predict_proba(test_data)[1]  # 概率

print(classification_report(y_true, y_pred))
print("AUC:", roc_auc_score(y_true, y_prob))


              precision    recall  f1-score   support

          -1       0.66      0.53      0.59       236
           0       0.01      0.20      0.02         5
           1       0.56      0.47      0.51       185

    accuracy                           0.50       426
   macro avg       0.41      0.40      0.37       426
weighted avg       0.61      0.50      0.55       426



ValueError: multi_class must be in ('ovo', 'ovr')

原始 basis_ml.ipynb 单元索引 36（从 0 开始）

In [6]:
predictor.path

'd:\\mycodelife\\workshop\\yinhe\\daily_factor\\AutogluonModels\\ag-20251015_014109'

原始 basis_ml.ipynb 单元索引 38（从 0 开始）

In [ ]:
import data
import new_factor as nf
import numpy as np
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
import factor
import re
import os, sys
from datetime import datetime

原始 basis_ml.ipynb 单元索引 39（从 0 开始）

In [ ]:
myfactor.get_data('tier2').to_excel('factor/tier2.xlsx')
df_raw = merge_excels_by_first_col('factor', 'merged.xlsx')

原始 basis_ml.ipynb 单元索引 40（从 0 开始）

In [ ]:
df_raw = pd.read_excel('merged.xlsx')
df_raw.set_index('date', inplace=True)

# 涨跌标签：下一期比当前期涨为 1，跌为 -1，不变为 0
df_raw['label'] = (df_raw['tier2__二级资本债_5Y'].shift(-1) - df_raw['tier2__二级资本债_5Y']).apply(
    lambda x: 1 if x > 0 else -1 if x < 0 else 0
)
# 删除最后一行（因为没有未来数据做标签）
df_clean = df_raw.iloc[:-1]
# 假设 df_clean 已经准备好，索引为时间或顺序

label_col = "label"
split_point = int(len(df_clean) * 0.8)

# 按顺序切分
df_train = df_clean.iloc[:split_point]
df_test  = df_clean.iloc[split_point:]

# 拆分特征和标签
X_train, y_train = df_train.drop(columns=[label_col]), df_train[label_col]
X_test,  y_test  = df_test.drop(columns=[label_col]),  df_test[label_col]

train_data = pd.concat([X_train,y_train],axis=1)
test_data = pd.concat([X_test, y_test],axis = 1)

原始 basis_ml.ipynb 单元索引 75（从 0 开始）

In [1]:
import data
import new_factor as nf
import numpy as np
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
import factor
import re
import os, sys
from datetime import datetime

# 把脚本所在目录设为当前工作目录
# os.chdir(os.path.dirname(os.path.abspath(__file__)))

def merge_excels_by_first_col(folder_path, output_file="merged.xlsx"):
    # 排除：以8位日期开头、紧跟“-因子清单”的文件（不含扩展名匹配）
    pattern = re.compile(r"^\d{8}-因子清单")
    files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".xlsx")
        and not f.startswith("~")
        and not pattern.match(os.path.splitext(f)[0])
    ]
    if not files:
        print("未找到任何可用的 .xlsx 文件")
        return
    dfs = []
    for f in sorted(files):
        path = os.path.join(folder_path, f)
        try:
            # 第一行是列名，第一列作为索引（合并键）
            df = pd.read_excel(path, header=0, index_col=0)
        except Exception as e:
            print(f"读取失败：{path}，原因：{e}")
            continue
        # 规范化索引（第一列），避免空白/类型不一致
        df.index = df.index.astype(str).str.strip()
        # 列名加前缀（保留原始列名，避免冲突）
        stem = os.path.splitext(f)[0]
        df.columns = [f"{stem}__{col}" for col in df.columns]
        dfs.append(df)
    if not dfs:
        print("没有成功读取的表格")
        return
    # 以索引（第一列）按“交集”对齐拼接
    merged = pd.concat(dfs, axis=1, join="inner")
    # 索引转为普通列，并命名为 "date"
    merged = merged.reset_index().rename(columns={"index": "date"})
    # 导出
    merged.to_excel(output_file, index=False, header=True)
    return merged
#data.run_data()
# factor.run_factor()
myfactor = nf.NewFactor()
myfactor.get_data('tier2').to_excel('factor/tier2.xlsx')
df_raw = merge_excels_by_first_col('factor', 'merged_tier2.xlsx')

d:\conda\envs\pavane\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


原始 basis_ml.ipynb 单元索引 76（从 0 开始）

In [ ]:
import pandas as pd

df_raw = pd.read_excel('merged_tier2.xlsx')
df_raw.set_index('date', inplace=True)

# 先删除目标列为空的行
df_raw = df_raw.dropna(subset=['tier2__二级资本债_5Y'])

# 涨跌标签：下一期比当前期涨为 1，跌为 -1，不变为 0
df_raw['label'] = (df_raw['tier2__二级资本债_5Y'].shift(-1) - df_raw['tier2__二级资本债_5Y']).apply(
    lambda x: -1 if x > 0 else (1 if x < 0 else 0)
)

# 若为 0，则继承前一个标签
df_raw['label'] = df_raw['label'].replace(0, method='ffill')

# 删除最后一行（因为没有未来数据做标签）
df_clean = df_raw.iloc[:-1]

# 假设 df_clean 已经准备好，索引为时间或顺序
label_col = "label"
split_point = int(len(df_clean) * 0.8)

# 按顺序切分
df_train = df_clean.iloc[:split_point]
df_test  = df_clean.iloc[split_point:]

# 拆分特征和标签
X_train, y_train = df_train.drop(columns=[label_col]), df_train[label_col]
X_test,  y_test  = df_test.drop(columns=[label_col]),  df_test[label_col]

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)